In [1]:
from dotenv import load_dotenv
import sys
import asyncio
import os
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain.agents import create_agent

from langchain.messages import HumanMessage
load_dotenv()

model=os.getenv("MODEL", "gpt-4o-mini")

In [6]:

# Fix for Windows issues in Jupyter notebooks
# if sys.platform == "win32":
#     # 1. Use ProactorEventLoop for subprocess support
#     if not isinstance(asyncio.get_event_loop_policy(), asyncio.WindowsProactorEventLoopPolicy):
#         asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())
    
#     # 2. Redirect stderr to avoid fileno() error when launching MCP servers
#     if "ipykernel" in sys.modules:
#         sys.stderr = sys.__stderr__


import asyncio
import sys

# Fix for Windows and Jupyter notebook issues
if sys.platform == "win32":
    # 1. Directly set ProactorEventLoopPolicy (avoids deprecated policy checks)
    asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())
    
    # 2. Redirect streams to avoid UnsupportedOperation: fileno when launching MCP servers
    if "ipykernel" in sys.modules:
        sys.stdout = sys.__stdout__
        sys.stderr = sys.__stderr__


## Local MCP server

In [9]:
client = MultiServerMCPClient(
    {
        "local_server": {
                "transport": "stdio",
                "command": "uv",
                "args": [
                    "run",
                    "python", 
                    "resources/2.1_mcp_server.py"],
            }
    }
)

In [10]:
# get tools
tools = await client.get_tools()

# get resources
resources = await client.get_resources("local_server")

# get prompts
prompt = await client.get_prompt("local_server", "prompt")
prompt = prompt[0].content

UnsupportedOperation: fileno

In [ ]:


agent = create_agent(
    model=model,
    tools=tools,
    system_prompt=prompt
)

In [ ]:


config = {"configurable": {"thread_id": "1"}}

response = await agent.ainvoke(
    {"messages": [HumanMessage(content="Tell me about the langchain-mcp-adapters library")]},
    config=config
)

In [ ]:
from pprint import pprint

pprint(response)

## Online MCP

In [11]:
client = MultiServerMCPClient(
    {
        "time": {
            "transport": "stdio",
            "command": "uv",
            "args": [
                "run",
                "python",
                "-m",
                "mcp_server_time",
                "--local-timezone=America/New_York"
            ]
        }
    }
)

tools = await client.get_tools()

UnsupportedOperation: fileno

In [ ]:
agent = create_agent(
    model=model,
    tools=tools,
)

In [ ]:
question = HumanMessage(content="What time is it?")

response = await agent.ainvoke(
    {"messages": [question]}
)

pprint(response)